# Module `models.py`

Ce notebook explique les structures de donnees centrales du projet CESIPATH.

Le but de ce module n'est pas de resoudre le probleme, mais de definir un vocabulaire commun pour les autres briques : statut des routes, couts statiques, configuration du generateur, instance complete et etat dynamique du reseau.

## Pourquoi commencer par les modeles ?

Avant de generer ou de resoudre un graphe, on a besoin d'objets clairs :

- `EdgeStatus` indique si une route est libre, surchargee ou interdite statiquement.
- `EdgeAttributes` stocke le cout de base, le statut et le surcout statique d'une vraie route.
- `GraphGenerationConfig` regroupe tous les parametres modifiables du generateur.
- `GraphInstance` represente une instance complete : graphe de base, graphe residuel, graphe complete et demandes.
- `DynamicGraphSnapshot` represente l'etat du reseau a un tour donne pendant la simulation dynamique.

Cette separation evite de melanger les parametres, les donnees generees et l'etat dynamique courant.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.models import EdgeAttributes, EdgeStatus, GraphGenerationConfig

## Configuration generale

`GraphGenerationConfig` est le point de controle principal.

Quelques variables importantes :

- `node_count` : nombre total de sommets, depot compris.
- `edge_density` : densite cible du graphe de base. Si elle vaut `None`, le profil automatique choisit une densite adaptee a la taille du graphe.
- `auto_density_profile` : active les bornes automatiques selon `node_count`.
- `forbidden_rate` : probabilite qu'une arete soit interdite statiquement lors de la generation.
- `surcharge_rate` : probabilite qu'une arete recoive un surcout statique.
- `dynamic_sigma` : amplitude des variations dynamiques.
- `dynamic_mean_reversion_strength` : force de rappel vers le cout statique.
- `dynamic_max_multiplier` : borne haute du cout dynamique par rapport au cout statique.
- `dynamic_forbid_probability` : probabilite qu'une route devienne temporairement indisponible.
- `dynamic_restore_probability` : probabilite qu'une route indisponible redevienne disponible.
- `dynamic_max_disabled_ratio` : part maximale de routes dynamiquement `OFF`.

In [ ]:
config = GraphGenerationConfig(
    node_count=8,
    edge_density=None,
    auto_density_profile=True,
    vehicle_capacity=45,
    dynamic_sigma=0.18,
    dynamic_mean_reversion_strength=0.35,
    dynamic_max_multiplier=1.8,
    dynamic_forbid_probability=0.03,
    dynamic_restore_probability=0.2,
    dynamic_max_disabled_ratio=0.2,
)

print(config)
print("Densite cible base :", config.resolved_edge_density)
print("Bornes densite base :", config.resolved_min_base_density, config.resolved_max_base_density)
print("Bornes densite residuelle :", config.resolved_min_residual_density, config.resolved_max_residual_density)
print("Degre moyen residuel minimal :", config.resolved_min_average_residual_degree)
print("Densite minimale dynamique :", config.resolved_dynamic_min_density)
print("Degre moyen minimal dynamique :", config.resolved_dynamic_min_average_degree)

## Lecture du retour

Les proprietes qui commencent par `resolved_` sont des valeurs deduites.

Par exemple, quand `edge_density=None`, le generateur ne prend pas une densite arbitraire. Il lit `node_count`, choisit un profil recommande, puis utilise le milieu de la plage comme densite cible.

Cela permet d'avoir des graphes petits plutot denses, et des graphes grands moins denses, ce qui evite de creer trop d'aretes inutiles.

## Exemple d'arete

Une arete possede :

- un `base_cost`, c'est le cout physique minimal de la route ;
- un `status`, qui peut etre `FREE`, `SURCHARGED` ou `FORBIDDEN` ;
- un `static_surcharge`, applique seulement si la route est surchargee.

Le `static_cost` est calcule automatiquement :

```text
static_cost = base_cost * (1 + static_surcharge)
```

Si l'arete est interdite, son `static_cost` devient `inf`.

In [ ]:
free_edge = EdgeAttributes(base_cost=10.0)
surcharged_edge = EdgeAttributes(base_cost=10.0, status=EdgeStatus.SURCHARGED, static_surcharge=0.25)
forbidden_edge = EdgeAttributes(base_cost=10.0, status=EdgeStatus.FORBIDDEN)

print("Arete libre :", free_edge.static_cost)
print("Arete surchargee :", surcharged_edge.static_cost)
print("Arete interdite :", forbidden_edge.static_cost)

## Pourquoi ces modeles sont utiles ?

Le generateur, la dynamique et la visualisation lisent tous les memes structures.

Ainsi, si on modifie une constante comme `dynamic_max_multiplier`, l'information circule proprement jusqu'au calcul dynamique sans devoir changer plusieurs fichiers manuellement.